# 心思维｜VocalParse 双模式受控对照

本 Notebook 接续 2026-11-22 的实验。使用同一个 VocalParse checkpoint、同一段原曲 30–55 秒音频、相同生成参数，分别运行 `audio-only` 和 `audio-lyric`。模型权重只加载一次。

## 重要边界

- 只比较两种提示模式对同一输入的影响，不判断哪一个结果就是真实乐谱。
- 输入的人声分离轨仍含混响、合唱/叠唱，音高输出只能辅助观察。
- 不运行 TorchCREPE，不自动安装依赖，不自动下载模型，不修改生产识别流程。
- 需要已有可用的 Colab GPU、VocalParse 代码/依赖、checkpoint 和音频文件。缺少任一项会明确停止，不擅自下载。

## 使用方式

运行唯一代码单元格，按提示粘贴 Drive 中的人声文件路径和 VocalParse checkpoint 路径。代码会自动截取 30–55 秒，并把对照 JSON 保存到 `MyDrive/XinsW_Lead_Transcription/VocalParse_Mode_Comparison/`。

**歌词提示**：代码预填了该 25 秒片段的歌词。运行时会出现歌词输入框，先听一下并确认字词；直接回车使用预填歌词，有误就把正确歌词粘贴进去。

In [ ]:
# VocalParse 双模式对照：同一音频、同一模型、两种提示
# 本单元格不会安装依赖、下载模型或运行 TorchCREPE。
# 它会加载你指定的本地 VocalParse checkpoint，并对同一段 25 秒音频执行两次推理。

from pathlib import Path
import json
import hashlib
import difflib
import gc
import numpy as np
import torch
from google.colab import drive

drive.mount("/content/drive")

# === 只需在下方两个输入框粘贴路径，不需要修改代码 ===
audio_path = input("已分离人声音频文件的完整路径（Drive 路径）: ").strip().strip('"')
checkpoint_path = input("VocalParse checkpoint 文件夹路径: ").strip().strip('"')

audio_file = Path(audio_path)
checkpoint_dir = Path(checkpoint_path)
if not audio_file.is_file():
    raise FileNotFoundError(f"找不到音频文件：{audio_file}")
if not checkpoint_dir.exists():
    raise FileNotFoundError(f"找不到 checkpoint 路径：{checkpoint_dir}")
if not (checkpoint_dir / "config.json").is_file():
    candidates = list(checkpoint_dir.glob("checkpoint-*/config.json"))
    if not candidates:
        raise FileNotFoundError(
            "指定目录中没有 config.json，也没有 checkpoint-*/config.json。"
            "请填写实际 checkpoint 文件夹，或包含 checkpoint-* 的训练输出目录。"
        )

# 25 秒片段：原曲 30–55 秒。两种模式严格使用同一段 PCM。
START_SEC = 30.0
END_SEC = 55.0
SAMPLE_RATE = 16000
MAX_NEW_TOKENS = 512
ATTN_IMPLEMENTATION = "sdpa"

# 这是本次对照使用的已知歌词（对应原曲 30–55 秒）。
# 若你听原音后确认有字词不同，请先改正下面这行再运行。
DEFAULT_GROUND_TRUTH_LYRICS = "如果世界变幻太快当青春慢慢走远当容颜不再如初见你不必问我爱不爱也不"
GROUND_TRUTH_LYRICS = input("请确认/粘贴 30–55 秒片段的歌词（直接回车使用预填歌词）: ").strip() or DEFAULT_GROUND_TRUTH_LYRICS

# 先确认上次运行环境中的 VocalParse 依赖是否仍在；缺失时停止，不自动安装/下载。
try:
    from vocalparse.api import VocalParseTranscriber
    from vocalparse.model import load_audio
    from vocalparse.evaluation import parse_transcription_text
except Exception as exc:
    raise RuntimeError(
        "当前 Colab 环境没有可用的 VocalParse 代码/依赖。"
        "本 Notebook 不会擅自安装或下载模型；请把完整报错发回来，我们再接着配置。"
    ) from exc

print("\n实验设置：")
print("音频：", audio_file)
print("Checkpoint：", checkpoint_dir)
print(f"原曲时间段：{START_SEC:.1f}–{END_SEC:.1f} 秒")
print("采样率：", SAMPLE_RATE)
print("已知歌词：", GROUND_TRUTH_LYRICS)
print("GPU：", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "不可用（本次先停止，避免误在 CPU 上长时间运行）")

if not torch.cuda.is_available():
    raise RuntimeError("当前运行时没有可用 GPU。未启动模型加载或推理，请切换到有 GPU 的 Colab 运行时后重试。")

# 使用 VocalParse 自己的音频加载器，避免两种模式的预处理出现差异。
full_audio = load_audio(str(audio_file), sr=SAMPLE_RATE).astype(np.float32)
start_sample = int(round(START_SEC * SAMPLE_RATE))
end_sample = int(round(END_SEC * SAMPLE_RATE))
if len(full_audio) < end_sample:
    raise ValueError(
        f"音频只有 {len(full_audio)/SAMPLE_RATE:.2f} 秒，"
        f"不足以截取 {START_SEC:.1f}–{END_SEC:.1f} 秒。"
    )
clip = np.ascontiguousarray(full_audio[start_sample:end_sample], dtype=np.float32)
if len(clip) != int((END_SEC - START_SEC) * SAMPLE_RATE):
    raise ValueError("截取后的音频长度不正确，已停止。")

audio_sha256 = hashlib.sha256(clip.tobytes()).hexdigest()
print(f"\n片段长度：{len(clip)/SAMPLE_RATE:.2f} 秒；SHA256：{audio_sha256}")

# 同一个 Transcriber 实例只加载一次权重；两种模式顺序运行，使用相同音频和生成参数。
transcriber = None
try:
    print("\n[1/2] 加载 VocalParse 模型……")
    transcriber = VocalParseTranscriber(
        checkpoint=str(checkpoint_dir),
        attn_implementation=ATTN_IMPLEMENTATION,
    )

    print("\n[2/2] 运行 audio-only……")
    raw_audio_only = transcriber.transcribe(
        [clip],
        sr=SAMPLE_RATE,
        max_new_tokens=MAX_NEW_TOKENS,
        batch_size=1,
        inference_mode="audio-only",
    )[0]

    print("运行 audio-lyric（提供已知歌词）……")
    raw_audio_lyric = transcriber.transcribe(
        [clip],
        sr=SAMPLE_RATE,
        max_new_tokens=MAX_NEW_TOKENS,
        batch_size=1,
        lyrics=[GROUND_TRUTH_LYRICS],
        inference_mode="audio-lyric",
    )[0]
finally:
    if transcriber is not None:
        transcriber.close()
        del transcriber
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 用同一个 VocalParse 官方解析函数解析两份原始输出。
parsed_audio_only = parse_transcription_text(raw_audio_only)
parsed_audio_lyric = parse_transcription_text(raw_audio_lyric)

def normalize_result(parsed):
    if not isinstance(parsed, dict):
        raise TypeError(f"解析结果不是 dict：{type(parsed).__name__}")
    required = ("bpm", "words", "pitches", "notes")
    missing = [key for key in required if key not in parsed]
    if missing:
        raise ValueError(f"解析结果缺少字段：{missing}")
    lengths = {key: len(parsed[key]) for key in ("words", "pitches", "notes")}
    if len(set(lengths.values())) != 1:
        raise ValueError(f"歌词/音高/时值数组长度不一致：{lengths}")
    return {
        "bpm": float(parsed["bpm"]),
        "words": [str(x) for x in parsed["words"]],
        "pitches": [int(x) for x in parsed["pitches"]],
        "notes": [str(x) for x in parsed["notes"]],
    }

a = normalize_result(parsed_audio_only)
b = normalize_result(parsed_audio_lyric)

# 先按词/字序列对齐，避免把插入或漏字之后的所有后续音符都误判为同一位置。
matcher = difflib.SequenceMatcher(a=a["words"], b=b["words"], autojunk=False)
alignment = []
for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    alignment.append({
        "operation": tag,
        "audio_only_range": [i1, i2],
        "audio_lyric_range": [j1, j2],
        "audio_only_words": a["words"][i1:i2],
        "audio_lyric_words": b["words"][j1:j2],
    })

# 逐索引表只作快速浏览；真正的歌词对齐请看上面的 SequenceMatcher 区段。
max_len = max(len(a["words"]), len(b["words"]))
index_rows = []
for i in range(max_len):
    row_a = i < len(a["words"])
    row_b = i < len(b["words"])
    index_rows.append({
        "index_1based": i + 1,
        "audio_only_word": a["words"][i] if row_a else None,
        "audio_lyric_word": b["words"][i] if row_b else None,
        "same_word_at_index": row_a and row_b and a["words"][i] == b["words"][i],
        "audio_only_pitch_midi": a["pitches"][i] if row_a else None,
        "audio_lyric_pitch_midi": b["pitches"][i] if row_b else None,
        "same_pitch_at_index": row_a and row_b and a["pitches"][i] == b["pitches"][i],
        "audio_only_duration": a["notes"][i] if row_a else None,
        "audio_lyric_duration": b["notes"][i] if row_b else None,
        "same_duration_at_index": row_a and row_b and a["notes"][i] == b["notes"][i],
    })

summary = {
    "comparison_type": "same_audio_same_checkpoint_audio_only_vs_audio_lyric",
    "ground_truth_claim": False,
    "audio_file": str(audio_file),
    "checkpoint": str(checkpoint_dir),
    "clip_start_sec": START_SEC,
    "clip_end_sec": END_SEC,
    "sample_rate": SAMPLE_RATE,
    "clip_sha256": audio_sha256,
    "known_lyrics_prompt": GROUND_TRUTH_LYRICS,
    "generation": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "batch_size": 1,
        "attention": ATTN_IMPLEMENTATION,
    },
    "audio_only": {
        "raw_text": raw_audio_only,
        "parsed": a,
        "event_count": len(a["words"]),
    },
    "audio_lyric": {
        "raw_text": raw_audio_lyric,
        "parsed": b,
        "event_count": len(b["words"]),
    },
    "summary": {
        "bpm_audio_only": a["bpm"],
        "bpm_audio_lyric": b["bpm"],
        "event_count_audio_only": len(a["words"]),
        "event_count_audio_lyric": len(b["words"]),
        "same_word_same_index_count": sum(r["same_word_at_index"] for r in index_rows),
        "same_pitch_same_index_count": sum(r["same_pitch_at_index"] for r in index_rows),
        "same_duration_same_index_count": sum(r["same_duration_at_index"] for r in index_rows),
        "lyric_sequence_alignment": alignment,
        "interpretation_warning": (
            "仅比较同一音频在两种提示模式下的输出差异；"
            "不是人工真值准确率。分离人声含混响/合唱/叠唱，"
            "不可将 TorchCREPE 或任一模型输出当作主唱真值。"
        ),
    },
    "index_comparison": index_rows,
}

output_dir = Path("/content/drive/MyDrive/XinsW_Lead_Transcription/VocalParse_Mode_Comparison")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "vocalparse_audio_only_vs_audio_lyric.json"
with output_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n===== 双模式对照摘要 =====")
print(json.dumps(summary["summary"], ensure_ascii=False, indent=2))
print("\n===== 两种模式原始解析结果 =====")
print("audio-only：", json.dumps(a, ensure_ascii=False))
print("audio-lyric：", json.dumps(b, ensure_ascii=False))
print("\n完整报告已保存：", output_path)
